## Pytorch MNIST classification

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from IPython.core.debugger import set_trace

In [3]:
image_folder = './images'

transformer = transforms.Compose([transforms.ToTensor(),
                                 transforms.Normalize((0.5,),(1.0,))])

train_set = datasets.MNIST(root=image_folder, train=True, transform=transformer, download=True)
test_set = datasets.MNIST(root=image_folder, train=False, transform=transformer, download=True)

batch_size = 64

print(train_set)

train_loader = torch.utils.data.DataLoader(dataset=train_set,
                                         batch_size=batch_size,
                                         shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_set,
                                         batch_size=batch_size,
                                         shuffle=False)



Processing...
Done!
Dataset MNIST
    Number of datapoints: 60000
    Split: train
    Root Location: ./images
    Transforms (if any): Compose(
                             ToTensor()
                             Normalize(mean=(0.5,), std=(1.0,))
                         )
    Target Transforms (if any): None


In [4]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 500)
        self.fc2 = nn.Linear(500, 256)
        self.fc3 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(-1, 28*28)
        x1 = F.relu(self.fc1(x))
        x2 = F.relu(self.fc2(x1))
        x3 = self.fc3(x2)
#         set_trace()
        return F.log_softmax(x3,1)
    
    def name(self):
        return "MLP"

In [ ]:
model = MLP()

optimizer = optim.SGD(model.parameters(), lr=0.01)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    correct_cnt, ave_loss = 0, 0
    total_cnt = 0
    for batch_idx, (x, target) in enumerate(train_loader):
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, target)
        _, pred_label = torch.max(out.data, 1)
        total_cnt += x.shape[0]
        correct_cnt+= (pred_label == target).sum().item()
        ave_loss = ave_loss * 0.9 + loss.item() * 0.1
        loss.backward()
        optimizer.step()
        if (batch_idx+1) % 100 == 0 or (batch_idx+1) == len(train_loader):
            print('==>>> epoch: {}, batch index: {}, train loss: {:.6f}, acc: {:.3f}'.format(
                epoch, batch_idx+1, ave_loss, correct_cnt*1.0/total_cnt))
    # testing
    correct_cnt, ave_loss = 0, 0
    total_cnt = 0
    for batch_idx, (x, target) in enumerate(test_loader):
        out = model(x)
        loss = criterion(out, target)
        _, pred_label = torch.max(out.data, 1)
        total_cnt += x.shape[0]
#         print(target.data)
        correct_cnt += (pred_label == target).sum().item()
        # smooth average
        ave_loss = ave_loss * 0.9 + loss.item() * 0.1
        
        if(batch_idx+1) % 100 == 0 or (batch_idx+1) == len(test_loader):
            print('==>>> epoch: {}, batch index: {}, test loss: {:.6f}, acc: {:.3f}'.format(
                epoch, batch_idx+1, ave_loss, correct_cnt * 1.0 / total_cnt))

==>>> epoch: 0, batch index: 100, train loss: 2.244679, acc: 0.241
==>>> epoch: 0, batch index: 200, train loss: 2.152687, acc: 0.307
==>>> epoch: 0, batch index: 300, train loss: 2.001786, acc: 0.374
==>>> epoch: 0, batch index: 400, train loss: 1.721265, acc: 0.426
==>>> epoch: 0, batch index: 500, train loss: 1.416741, acc: 0.474
==>>> epoch: 0, batch index: 600, train loss: 1.133527, acc: 0.515
==>>> epoch: 0, batch index: 700, train loss: 0.932821, acc: 0.552
==>>> epoch: 0, batch index: 800, train loss: 0.795490, acc: 0.584
==>>> epoch: 0, batch index: 900, train loss: 0.699649, acc: 0.611
==>>> epoch: 0, batch index: 938, train loss: 0.689511, acc: 0.620
==>>> epoch: 0, batch index: 100, test loss: 0.550216, acc: 0.825
==>>> epoch: 0, batch index: 157, test loss: 0.629890, acc: 0.843
==>>> epoch: 1, batch index: 100, train loss: 0.608450, acc: 0.840
==>>> epoch: 1, batch index: 200, train loss: 0.554067, acc: 0.845
==>>> epoch: 1, batch index: 300, train loss: 0.503745, acc: 0.8